In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 1


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')
dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Donovan Mitchell,Over,27.5,-137,2025-11-14,2025-11-13T18:04:13Z
1,PrizePicks,player_points,Donovan Mitchell,Under,27.5,-137,2025-11-14,2025-11-13T18:04:13Z
2,PrizePicks,player_points,Brandon Ingram,Over,20.0,-137,2025-11-14,2025-11-13T18:04:13Z
3,PrizePicks,player_points,Brandon Ingram,Under,20.0,-137,2025-11-14,2025-11-13T18:04:13Z
4,PrizePicks,player_points,Evan Mobley,Over,19.5,-137,2025-11-14,2025-11-13T18:04:13Z


### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Applications/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 6 teams with confirmed lineups


### Top EVs for single bets

In [7]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                             edge_threshold=0.20, stake=10, 
                             variance_inflation=1.1, 
                             use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(10)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 43 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Jarrett Allen,Bovada,17.5,20.73,Over,170,1,8.15,81.5,0.479,High
1,De'Andre Hunter,Bovada,21.5,23.24,Over,190,1,7.25,72.5,0.382,High
2,Jarrett Allen,Bovada,16.5,20.73,Over,135,1,6.77,67.7,0.502,High
3,De'Andre Hunter,Bovada,20.5,23.24,Over,155,1,6.35,63.5,0.410,High
4,Grayson Allen,Bovada,20.5,21.03,Over,205,1,6.23,62.3,0.304,High


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1,
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 29 players...
Processing 27 players with valid predictions...
Generated 295 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Scottie Barnes,Evan Mobley,17.5,19.5,over,over,1,17.0,0.85,High,High
1,Lauri Markkanen,Andrew Nembhard,25.5,16.5,over,over,1,17.0,0.85,High,High
2,Keyonte George,Grayson Allen,19.5,16.5,over,over,1,17.0,0.85,High,High
3,Keyonte George,T.J. McConnell,19.5,9.5,over,over,1,17.0,0.85,High,High
4,Immanuel Quickley,Ben Sheppard,15.5,5.5,over,over,1,17.0,0.85,High,High


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1,
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 43 players...
Processing 41 players with valid predictions...
Generated 697 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Donovan Mitchell,Brandon Ingram,27.5,20.0,over,over,1,17.0,0.85,High,High
1,Immanuel Quickley,Svi Mykhailiuk,15.5,9.5,over,over,1,17.0,0.85,High,High
2,Scottie Barnes,Isaiah Jackson,17.5,8.0,over,over,1,17.0,0.85,High,High
3,Scottie Barnes,Ryan Dunn,17.5,7.5,over,over,1,17.0,0.85,High,High
4,Scottie Barnes,Ben Sheppard,17.5,6.0,over,over,1,17.0,0.85,High,High


## 3 leg parlay

### Underdog picks

In [13]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1, 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 29 players...
Processing 27 players with valid predictions...
Generated 2849 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Jarrett Allen,De'Andre Hunter,Keyonte George,14.5,18.5,19.5,over,over,over,0,10.64,0.213,High,High,High
1,Jarrett Allen,De'Andre Hunter,Grayson Allen,14.5,18.5,16.5,over,over,over,0,10.43,0.209,High,High,High
2,Jarrett Allen,Sam Merrill,Keyonte George,14.5,10.5,19.5,over,over,over,0,10.34,0.207,High,High,High
3,Jarrett Allen,Sam Merrill,Grayson Allen,14.5,10.5,16.5,over,over,over,0,10.15,0.203,High,High,High
4,Scottie Barnes,Jarrett Allen,De'Andre Hunter,17.5,14.5,18.5,over,over,over,0,10.06,0.201,High,High,High


### Prizepicks picks

In [14]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=100, 
                     variance_inflation=1.1, 
                     use_monte_carlo=False, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 43 players...
Processing 41 players with valid predictions...
Generated 10448 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Jarrett Allen,De'Andre Hunter,Keyonte George,14.5,18.5,19.5,over,over,over,0,10.64,0.213,High,High,High
1,Jarrett Allen,De'Andre Hunter,Grayson Allen,14.5,18.5,16.5,over,over,over,0,10.43,0.209,High,High,High
2,Jarrett Allen,Sam Merrill,Keyonte George,14.5,10.5,19.5,over,over,over,0,10.34,0.207,High,High,High
3,Jarrett Allen,Sam Merrill,Grayson Allen,14.5,10.5,16.5,over,over,over,0,10.15,0.203,High,High,High
4,Scottie Barnes,Jarrett Allen,De'Andre Hunter,17.5,14.5,18.5,over,over,over,0,10.06,0.201,High,High,High
